# Meridian Guard 1.1 by TEKIMAX: deploy and call

Meridian Guard reads what an AI agent is about to do (the organization's policy, trusted facts, the request and the
proposed tool call) and recommends the next step from a fixed set of choices. It is advice, not compliance: your own
controls decide, and anything other than a clear proceed answer, including an error or a timeout, should be treated as
escalate (stop and ask a person).

This notebook:
1. deploys the model package you subscribed to on one `ml.c5.xlarge` real-time endpoint,
2. sends the two sample requests in `sample-request.json` (an allowed change inside the organization and an action
   that must stop),
3. maps each answer to proceed, verify or escalate,
4. deletes the endpoint and the model.

**Cost:** the software is free; you pay AWS for the `ml.c5.xlarge` endpoint while it exists (see SageMaker pricing for
your Region). Run the last cell when you are done.

## Prerequisites

- A subscription to **Meridian Guard 1.1 by TEKIMAX** in AWS Marketplace. After subscribing, open the product's
  configuration page and copy the model package ARN for your Region.
- A SageMaker execution role (in SageMaker Studio or a notebook instance, `get_execution_role()` returns it).
- Python packages: `sagemaker` (v2) and `boto3`.

In [ ]:
import json
import time

import boto3
import sagemaker
from sagemaker import ModelPackage

# Paste the model package ARN for your Region from the product's configuration page.
MODEL_PACKAGE_ARN = "<model package ARN for your Region>"

session = sagemaker.Session()
region = session.boto_region_name
role = sagemaker.get_execution_role()
endpoint_name = f"meridian-guard-1-1-{int(time.time())}"
print(region, endpoint_name)

## Deploy

One `ml.c5.xlarge` instance. The container runs with no network access. Deployment takes a few minutes.

In [ ]:
model = ModelPackage(role=role, model_package_arn=MODEL_PACKAGE_ARN, sagemaker_session=session)
model.deploy(initial_instance_count=1, instance_type="ml.c5.xlarge", endpoint_name=endpoint_name)

## The request

Each request carries:

| Field | What it is |
|---|---|
| `question` | Always `Which declared action should the agent take next?` |
| `state.policy` | The organization's rules, then `\nTRUSTED APPLICATION FACTS: ` and the facts your own systems confirm (identity, ownership, approvals on record). Never facts a user merely claims. |
| `state.conversation` | The request. When the agent proposes a tool call, end the user turn with `\n\nProposed next tool call: ` and the call as JSON with sorted keys (`json.dumps(call, ensure_ascii=False, sort_keys=True)`); this is the form the model reads. |
| `state.available_tools` | The tools the agent may use: `name` and `description`. |
| `criteria` | The choices, label to description (label set 1.1: nine labels, including `run_action`). |
| `proposed_call` | The same call as an object, `{"name": ..., "arguments": {...}}`. Without it the answer is never `run_action`. |
| `org_domain` | Optional: the organization's own mail domain, so an internal email can be recognized. |

The answer's `predicted_label` is one of the `criteria` labels; `candidates` ranks every choice by score.

In [ ]:
runtime = boto3.client("sagemaker-runtime", region_name=region)

PROCEED = {"answer", "clarify", "search_workspace", "search_public", "prepare_draft", "run_action"}
VERIFY = {"inspect_evidence"}


def decision_for(label):
    """proceed, verify or escalate. Anything unknown or missing escalates."""
    return "proceed" if label in PROCEED else "verify" if label in VERIFY else "escalate"


def decide(request):
    """Calls the endpoint. Any error or timeout is escalate, never proceed."""
    try:
        out = runtime.invoke_endpoint(EndpointName=endpoint_name, ContentType="application/json",
                                      Accept="application/json", Body=json.dumps(request))
        answer = json.loads(out["Body"].read())
    except Exception as error:  # noqa: BLE001
        return {"predicted_label": None, "decision": "escalate", "error": type(error).__name__}
    answer["decision"] = decision_for(answer.get("predicted_label"))
    return answer


with open("sample-request.json") as f:
    samples = json.load(f)
for name, request in samples.items():
    answer = decide(request)
    print(f"{name}: {answer.get('predicted_label')} -> {answer['decision']}")

Expected (`sample-response.json` has the full answers):

- `allowed_in_org_change`: `block_action`, `escalated: true`, so escalate. `run_action` scores highest, but
  `block_action` scores close to it, so this is a close call and it goes to a person.
- `must_stop_unrecorded_approval`: `request_approval`, escalate. The email leaves the organization and claims an
  approval that is not on record.

When a stop scores close to the top answer, the service chooses the stop and sends it to a person.

The `latency_ms` values in `sample-response.json` were recorded on TEKIMAX's serverless endpoint, where a first call
can take several seconds while the endpoint warms up.

What your agent does with each decision:

| Decision | Labels | What the agent does |
|---|---|---|
| proceed | `answer`, `clarify`, `search_workspace`, `search_public`, `prepare_draft`, `run_action` | Take the step. `run_action`: carry out the proposed call and record it to the person. |
| verify | `inspect_evidence` | Only read the approval and ownership records, then ask again with what was found in the trusted facts. |
| escalate | `request_approval`, `block_action`, any error or timeout | Stop and ask a person. |

**Callers built for label set 1.0** (eight labels, no `run_action`): send the eight 1.0 labels in `criteria` and leave
out `proposed_call`; the answer is always one of the labels you sent.

## Clean up

Deletes the endpoint, its configuration and the model so nothing keeps running.

In [ ]:
sm = boto3.client("sagemaker", region_name=region)
config = sm.describe_endpoint(EndpointName=endpoint_name)["EndpointConfigName"]
sm.delete_endpoint(EndpointName=endpoint_name)
sm.delete_endpoint_config(EndpointConfigName=config)
model.delete_model()
print("deleted", endpoint_name)